In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv('TCS_New.csv')

# Remove extra rows
df = df.iloc[2:].reset_index(drop=True)

df.columns = ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

df['Date'] = pd.to_datetime(df['Date'])
df['Close'] = df['Close'].astype(float)

df = df.sort_values('Date')

In [ ]:
# Returns
df['Returns'] = df['Close'].pct_change()

# Moving averages
df['MA10'] = df['Close'].rolling(10).mean()
df['MA20'] = df['Close'].rolling(20).mean()

# Exponential averages
df['EMA10'] = df['Close'].ewm(span=10).mean()
df['EMA20'] = df['Close'].ewm(span=20).mean()

# Momentum
df['Momentum'] = df['Close'] - df['Close'].shift(5)

# Volatility
df['Volatility'] = df['Returns'].rolling(10).std()

# 🎯 Target: 3-day direction (better than 1-day)
df['Direction'] = (df['Close'].shift(-3) > df['Close']).astype(int)

# Drop NaNs
df = df.dropna()

In [ ]:
features = df[['Returns', 'MA10', 'MA20', 'EMA10', 'EMA20', 'Momentum', 'Volatility']].values
target = df['Direction'].values

In [ ]:
split = int(0.8 * len(features))

train_data = features[:split]
test_data = features[split:]

y_train_raw = target[:split]
y_test_raw = target[split:]

In [ ]:
scaler = MinMaxScaler()

train_data = scaler.fit_transform(train_data)
test_data = scaler.transform(test_data)

In [ ]:
def create_dataset(data, target, time_step=60):
    X, y = [], []
    for i in range(time_step, len(data)):
        X.append(data[i-time_step:i])
        y.append(target[i])
    return np.array(X), np.array(y)

X_train, y_train = create_dataset(train_data, y_train_raw, 60)
X_test, y_test = create_dataset(test_data, y_test_raw, 60)

In [ ]:
print(X_train.shape)  # (samples, 60, features)

(913, 60, 7)


In [ ]:
model = Sequential()

model.add(Bidirectional(LSTM(100, return_sequences=True),
                        input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.3))

model.add(Bidirectional(LSTM(100)))
model.add(Dropout(0.3))

model.add(Dense(50, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
early_stop = EarlyStopping(patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 7s 45ms/step - accuracy: 0.5016 - loss: 0.6984 - val_accuracy: 0.5326 - val_loss: 0.6909
Epoch 2/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.5312 - loss: 0.6880 - val_accuracy: 0.4185 - val_loss: 0.7243
Epoch 3/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.5279 - loss: 0.6907 - val_accuracy: 0.5598 - val_loss: 0.6842
Epoch 4/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5312 - loss: 0.6868 - val_accuracy: 0.4402 - val_loss: 0.6977
Epoch 5/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5301 - loss: 0.6839 - val_accuracy: 0.5924 - val_loss: 0.6931
Epoch 6/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5422 - loss: 0.6813 - val_accuracy: 0.5761 - val_loss: 0.6808
Epoch 7/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5553 - loss: 0.6737 - val_accuracy: 0.5380 - val_loss: 0.7181
Epoch 8/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5553 - loss: 0.6853 - val_accuracy: 0.5489 - v

In [ ]:
probs = model.predict(X_test)
threshold = np.mean(probs)   # dynamic threshold

preds = (probs > threshold).astype(int).flatten()

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


In [ ]:
probs = model.predict(X_test)

# only take strong signals
buy_signals = probs > 0.55
sell_signals = probs < 0.45

In [ ]:
print("Accuracy:", accuracy_score(y_test, preds))
print("Precision:", precision_score(y_test, preds))
print("Recall:", recall_score(y_test, preds))

Accuracy: 0.6141304347826086
Precision: 0.5384615384615384
Recall: 0.5454545454545454


In [ ]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df['RSI'] = compute_rsi(df['Close'])

In [ ]:
ema12 = df['Close'].ewm(span=12).mean()
ema26 = df['Close'].ewm(span=26).mean()

df['MACD'] = ema12 - ema26
df['MACD_signal'] = df['MACD'].ewm(span=9).mean()

In [ ]:
df['BB_mean'] = df['Close'].rolling(20).mean()
df['BB_std'] = df['Close'].rolling(20).std()

df['BB_upper'] = df['BB_mean'] + 2 * df['BB_std']
df['BB_lower'] = df['BB_mean'] - 2 * df['BB_std']

In [ ]:
features = df[
    ['Returns','MA10','MA20','EMA10','EMA20',
     'Momentum','Volatility',
     'RSI','MACD','MACD_signal',
     'BB_upper','BB_lower']
].values

In [ ]:
df['Direction'] = (df['Close'].shift(-5) > df['Close']).astype(int)

In [ ]:
time_step = 90

In [ ]:
model = Sequential()

model.add(Bidirectional(LSTM(128, return_sequences=True),
                        input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.3))

model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_scheduler = ReduceLROnPlateau(patience=3, factor=0.5)

model.fit(
    X_train, y_train,
    epochs=60,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, lr_scheduler]
)

Epoch 1/60
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - accuracy: 0.5016 - loss: 0.6957 - val_accuracy: 0.4728 - val_loss: 0.6951 - learning_rate: 0.0010
Epoch 2/60
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5104 - loss: 0.6917 - val_accuracy: 0.5543 - val_loss: 0.6867 - learning_rate: 0.0010
Epoch 3/60
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5214 - loss: 0.6938 - val_accuracy: 0.5543 - val_loss: 0.6858 - learning_rate: 0.0010
Epoch 4/60
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5268 - loss: 0.6912 - val_accuracy: 0.5543 - val_loss: 0.6805 - learning_rate: 0.0010
Epoch 5/60
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5290 - loss: 0.6875 - val_accuracy: 0.4185 - val_loss: 0.7242 - learning_rate: 0.0010


In [ ]:
probs = model.predict(X_test).flatten()

preds = (probs > 0.50).astype(int)

strong_mask = (probs > 0.50) | (probs < 0.49)

filtered_preds = preds[strong_mask]
filtered_actual = y_test[strong_mask]

print("Filtered Accuracy:", accuracy_score(filtered_actual, filtered_preds))
print("Number of trades:", len(filtered_preds))

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Filtered Accuracy: 0.4342105263157895
Number of trades: 152


In [ ]:
print("Min:", probs.min())
print("Max:", probs.max())

Min: 0.4967487
Max: 0.52150834
